# 06 · Do dataset ao sistema rodando

Rodar **em casa**, uma vez. Ao final deste notebook você tem:

1. um **dataset de garrafas** baixado de um banco público (sem fotografar nada)
2. um **detector treinado** por você
3. as **inferências rodando** com esses pesos — em foto e **ao vivo**
4. um **classificador aberta/lacrada**, rotulado quase todo pela máquina
5. o **pipeline completo**: detecta → classifica → conta o estoque

A parte 4 é a que costuma matar projeto de visão: alguém tem que dizer, foto a
foto, qual é qual. Aqui isso é feito por um modelo que **já entende linguagem**
— e sobra para você só a revisão do que ele errou.

In [ ]:
# ── 1. instala a biblioteca e monta o Google Drive ──
%pip install -q ultralytics
from google.colab import drive
drive.mount('/content/drive')

from ultralytics import YOLO
import ultralytics, torch, os, glob
ultralytics.checks()
print("GPU disponivel:", torch.cuda.is_available())

In [ ]:
# ── ajuste de PALCO: tudo grande, porque a sala enxerga de 6 a 10 m ──
import matplotlib
matplotlib.rcParams.update({
    "figure.figsize": (16, 9),
    "figure.dpi": 110,
    "font.size": 22,
    "axes.titlesize": 30,
    "axes.labelsize": 24,
    "xtick.labelsize": 20,
    "ytick.labelsize": 20,
    "legend.fontsize": 22,
    "axes.grid": True,
    "grid.alpha": .25,
    "axes.facecolor": "#0d1117",
    "figure.facecolor": "#0d1117",
    "text.color": "#e6edf3",
    "axes.labelcolor": "#e6edf3",
    "xtick.color": "#e6edf3",
    "ytick.color": "#e6edf3",
    "axes.edgecolor": "#30363d",
    "axes.titlecolor": "#3fe0a8",
})
VERDE, VERMELHO, CINZA = "#3fe0a8", "#ff5c5c", "#7d8590"
DRIVE = "/content/drive/MyDrive/PALESTRA-IA"
print("palco configurado · raiz no Drive:", DRIVE)

---
## Parte 1 · O dataset, de graça

O **Open Images** é um banco público do Google com milhões de fotos **já
anotadas**. Dá para puxar só a classe que interessa.

`Bottle` é o guarda-chuva e rende mais caixas úteis; `Wine` traz contexto de
adega e mesa posta. Comece com 600–1000 imagens — mais que isso demora e não
muda o resultado da demo.

In [ ]:
%pip install -q fiftyone
import fiftyone as fo
import fiftyone.zoo as foz
import fiftyone.utils.random as four
print("fiftyone", fo.__version__)

In [ ]:
CLASSES = ["Bottle", "Wine"]      # ← mexa aqui
QUANTAS = 800                      # ← e aqui

dados = foz.load_zoo_dataset(
    "open-images-v7", split="train",
    label_types=["detections"], classes=CLASSES,
    max_samples=QUANTAS, only_matching=True,
    dataset_name="garrafas-palestra", overwrite=True,
)
print(dados)

In [ ]:
# ── exporta no formato do YOLO, direto no Drive ──
import os
DESTINO = f"{DRIVE}/04-garrafas/dataset-yolo"
os.makedirs(DESTINO, exist_ok=True)

four.random_split(dados, {"train": 0.8, "val": 0.2})
for split in ("train", "val"):
    dados.match_tags(split).export(
        export_dir=DESTINO, dataset_type=fo.types.YOLOv5Dataset,
        label_field="ground_truth", split=split, classes=CLASSES,
    )
print("exportado para", DESTINO)
print(sorted(os.listdir(DESTINO)))

---
## Parte 2 · Treinar o detector

Trinta épocas bastam para a demo. O modelo pronto já detecta `bottle`; este
treino serve para ter um detector **especializado em garrafa de vinho**, que o
COCO não separa de garrafa de água.

In [ ]:
EPOCAS = 30

det = YOLO(f"{DRIVE}/00-pesos/yolo11n.pt")
res = det.train(
    data=f"{DESTINO}/dataset.yaml",
    epochs=EPOCAS, imgsz=640, batch=16,
    project="/content/runs", name="garrafas-det", exist_ok=True,
    verbose=False, plots=True,
)

# guarda os pesos no Drive: treinou uma vez, usa para sempre
import shutil
PESOS_DET = f"{DRIVE}/04-garrafas/pesos/detector_garrafas.pt"
shutil.copy(f"{res.save_dir}/weights/best.pt", PESOS_DET)
print("detector salvo em", PESOS_DET)

In [ ]:
# ── como ele aprendeu ──
from IPython.display import Image, display
import os
for nome in ("results.png", "confusion_matrix_normalized.png"):
    caminho = os.path.join(str(res.save_dir), nome)
    if os.path.exists(caminho):
        print(nome); display(Image(filename=caminho, width=1100))

---
## Parte 3 · **Usando os pesos que você acabou de treinar**

Treinar sem testar não prova nada. Aqui o detector novo roda em imagens que
ele nunca viu, e você vê o resultado com os próprios olhos.

In [ ]:
# ── inferência em lote, nas imagens de validação ──
import glob, cv2, matplotlib.pyplot as plt, random

meu_detector = YOLO(PESOS_DET)

amostras = glob.glob(f"{DESTINO}/images/val/*")
amostras = [a for a in amostras if a.lower().endswith((".jpg", ".jpeg", ".png"))]
random.shuffle(amostras)
amostras = amostras[:6]

fig, axs = plt.subplots(2, 3, figsize=(22, 12))
total = 0
for ax, arq in zip(axs.ravel(), amostras):
    r = meu_detector.predict(arq, conf=.35, verbose=False)[0]
    total += len(r.boxes)
    ax.imshow(cv2.cvtColor(r.plot(line_width=3), cv2.COLOR_BGR2RGB))
    ax.set_title(f"{len(r.boxes)} garrafas", fontsize=24)
    ax.axis("off")
fig.suptitle(f"Detector treinado por você · {total} garrafas em 6 fotos novas", fontsize=32)
plt.tight_layout(); plt.show()

In [ ]:
# ── comparação honesta: o pronto x o seu ──
pronto = YOLO(f"{DRIVE}/00-pesos/yolo11n.pt")
arq = amostras[0]

fig, axs = plt.subplots(1, 2, figsize=(22, 9))
for ax, (modelo, titulo, kw) in zip(axs, [
        (pronto, "MODELO PRONTO (classe 'bottle' do COCO)", {"classes": [39]}),
        (meu_detector, "O QUE VOCÊ TREINOU", {})]):
    r = modelo.predict(arq, conf=.35, verbose=False, **kw)[0]
    ax.imshow(cv2.cvtColor(r.plot(line_width=3), cv2.COLOR_BGR2RGB))
    ax.set_title(f"{titulo} · {len(r.boxes)}", fontsize=22)
    ax.axis("off")
plt.tight_layout(); plt.show()

### 🔴 E ao vivo, com o seu detector

A mesma câmera das outras demos, agora rodando **o modelo que você treinou**.

In [ ]:
# ── motor de webcam ao vivo (leia o comentário: é o truque da demo) ──
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import cv2, numpy as np, PIL.Image, io, time

def iniciar_webcam(largura=640, altura=480):
    # cria o video no navegador + a camada de overlay por cima dele
    display(Javascript('''
      var video, div = null, stream, imgElement, labelElement, captureCanvas;
      var pendingResolve = null, shutdown = false;
      var LARG = %d, ALT = %d;

      function removeDom() {
        if (stream) stream.getVideoTracks()[0].stop();
        if (video) video.remove();
        if (div) div.remove();
        video = null; div = null; stream = null;
        imgElement = null; captureCanvas = null; labelElement = null;
      }

      function onAnimationFrame() {
        if (!shutdown) window.requestAnimationFrame(onAnimationFrame);
        if (pendingResolve) {
          var result = "";
          if (!shutdown) {
            captureCanvas.getContext('2d').drawImage(video, 0, 0, LARG, ALT);
            result = captureCanvas.toDataURL('image/jpeg', 0.75);
          }
          var lp = pendingResolve;
          pendingResolve = null;
          lp(result);
        }
      }

      async function criarDom() {
        if (div !== null) return stream;

        div = document.createElement('div');
        div.style.border = '2px solid #3fe0a8';
        div.style.padding = '3px';
        div.style.width = '100%%';
        div.style.maxWidth = '900px';
        div.style.borderRadius = '10px';
        document.body.appendChild(div);

        var parar = document.createElement('div');
        parar.innerHTML = '&#9632; clique aqui para encerrar';
        parar.style.cssText = 'cursor:pointer;background:#3fe0a8;color:#06231a;' +
          'font-weight:700;padding:10px 16px;border-radius:8px;text-align:center;' +
          'font-family:system-ui,sans-serif;font-size:18px';
        div.appendChild(parar);
        parar.onclick = function() { shutdown = true; };

        video = document.createElement('video');
        video.style.display = 'block';
        video.style.width = '100%%';
        video.setAttribute('playsinline', '');
        video.onclick = function() { shutdown = true; };

        stream = await navigator.mediaDevices.getUserMedia(
          {video: {width: LARG, height: ALT}});
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();

        // a camada que recebe o resultado do modelo, por cima do vídeo
        imgElement = document.createElement('img');
        imgElement.style.position = 'absolute';
        imgElement.style.zIndex = 1;
        imgElement.style.pointerEvents = 'none';
        imgElement.onclick = function() { shutdown = true; };
        div.appendChild(imgElement);

        labelElement = document.createElement('div');
        labelElement.style.cssText = 'font-family:system-ui,sans-serif;' +
          'font-size:20px;color:#e6edf3;padding:8px 4px';
        div.appendChild(labelElement);

        captureCanvas = document.createElement('canvas');
        captureCanvas.width = LARG;
        captureCanvas.height = ALT;
        window.requestAnimationFrame(onAnimationFrame);

        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        return stream;
      }

      async function quadro(rotulo, overlay) {
        if (shutdown) { removeDom(); shutdown = false; return ''; }
        stream = await criarDom();
        if (rotulo != "") labelElement.innerHTML = rotulo;
        if (overlay != "") {
          var r = video.getClientRects()[0];
          imgElement.style.top = r.top + "px";
          imgElement.style.left = r.left + "px";
          imgElement.style.width = r.width + "px";
          imgElement.style.height = r.height + "px";
          imgElement.src = overlay;
        }
        var result = await new Promise(function(resolve) { pendingResolve = resolve; });
        shutdown = false;
        return {'img': result};
      }
    ''' % (largura, altura)))


def _para_imagem(resposta):
    # base64 do navegador -> imagem BGR do OpenCV
    if not resposta:
        return None
    dados = b64decode(resposta.split(',')[1])
    arr = np.frombuffer(dados, dtype=np.uint8)
    return cv2.imdecode(arr, flags=1)


def _para_overlay(rgba):
    # array RGBA -> data URI PNG, para o navegador sobrepor ao video
    img = PIL.Image.fromarray(rgba, 'RGBA')
    buf = io.BytesIO()
    img.save(buf, format='png')
    return 'data:image/png;base64,' + b64encode(buf.getvalue()).decode('utf-8')


def rodar_ao_vivo(processa, largura=640, altura=480, rotulo_inicial='iniciando…'):
    # Laco principal.
    #
    # `processa(frame_bgr, overlay_rgba)` recebe o quadro e uma tela RGBA
    # transparente do mesmo tamanho, desenha nela, e devolve o texto do
    # painel. Encerre clicando no botão verde (ou no próprio vídeo).
    iniciar_webcam(largura, altura)
    overlay = np.zeros([altura, largura, 4], dtype=np.uint8)
    envio = ''
    rotulo = rotulo_inicial
    n = 0
    t0 = time.time()
    try:
        while True:
            resposta = eval_js('quadro("{}", "{}")'.format(rotulo, envio))
            if not resposta:
                break
            frame = _para_imagem(resposta['img'])
            if frame is None:
                break

            overlay[:] = 0
            texto = processa(frame, overlay)

            n += 1
            fps = n / max(1e-6, time.time() - t0)
            rotulo = '{} &nbsp;·&nbsp; {:.1f} quadros/s'.format(texto, fps)
            envio = _para_overlay(overlay)
    except Exception as e:
        print('encerrado:', type(e).__name__, e)
    print('fim · {} quadros processados'.format(n))

In [ ]:
import cv2, numpy as np

vistos_garrafa = set()

def processa_meu_detector(frame, overlay):
    h, w = frame.shape[:2]
    r = meu_detector.track(frame, persist=True, conf=.35, verbose=False, imgsz=480)[0]

    na_cena = 0
    if r.boxes is not None and len(r.boxes):
        na_cena = len(r.boxes)
        ids = (r.boxes.id.cpu().numpy().astype(int) if r.boxes.id is not None
               else np.arange(na_cena))
        for gid, caixa in zip(ids, r.boxes.xyxy.cpu().numpy().astype(int)):
            vistos_garrafa.add(int(gid))
            x1, y1, x2, y2 = caixa
            cv2.rectangle(overlay, (x1, y1), (x2, y2), (63, 224, 168, 255), 3)
            cv2.rectangle(overlay, (x1, max(0, y1 - 30)), (x1 + 74, y1), (63, 224, 168, 235), -1)
            cv2.putText(overlay, "#{}".format(gid), (x1 + 6, max(20, y1 - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (10, 15, 20, 255), 2)

    cv2.rectangle(overlay, (0, 0), (w, 92), (13, 17, 23, 215), -1)
    cv2.putText(overlay, "ESTOQUE", (16, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (125, 133, 144, 255), 2)
    cv2.putText(overlay, str(len(vistos_garrafa)), (16, 82),
                cv2.FONT_HERSHEY_SIMPLEX, 1.9, (63, 224, 168, 255), 4)
    cv2.putText(overlay, "NA CENA", (int(w * .45), 30), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (125, 133, 144, 255), 2)
    cv2.putText(overlay, str(na_cena), (int(w * .45), 82),
                cv2.FONT_HERSHEY_SIMPLEX, 1.9, (230, 237, 243, 255), 4)

    return "seu detector &nbsp;·&nbsp; estoque <b>{}</b> &nbsp;·&nbsp; na cena <b>{}</b>".format(
        len(vistos_garrafa), na_cena)

vistos_garrafa.clear()
rodar_ao_vivo(processa_meu_detector, largura=640, altura=480,
              rotulo_inicial='mostre uma garrafa…')

---
## Parte 4 · Rotular **sem** separar foto por foto

Agora o problema de verdade: o Open Images sabe que **é uma garrafa**. Não sabe
se está **aberta**. Isso ninguém tem pronto — e é justamente o que vale.

Arrastar centenas de recortes no Drive é inviável. A estratégia aqui é outra:

| passo | quem faz | esforço |
|---|---|---|
| recortar cada garrafa | o detector | zero |
| dar o primeiro palpite | **CLIP**, que entende texto e imagem | zero |
| corrigir o que ele errou | você, digitando números | minutos |
| montar train/ e val/ | o notebook | zero |

**CLIP** foi treinado com centenas de milhões de pares imagem+legenda. Ele não
conhece o seu problema, mas entende português e inglês o bastante para separar
*"garrafa lacrada, com cápsula no gargalo"* de *"garrafa aberta, sem rolha"* —
e acerta a maioria. Você vira revisor, não anotador.

In [ ]:
# ── recorta cada garrafa das imagens baixadas ──
import glob, os, cv2

RECORTES = f"{DRIVE}/04-garrafas/recortes"
os.makedirs(RECORTES, exist_ok=True)

# Este notebook escreve DENTRO do seu Drive. Nada é apagado sem você
# pedir: se já houver recortes de uma rodada anterior, a célula para e
# avisa, em vez de sobrescrever em silêncio.
APAGAR_ANTERIORES = False        # ← mude para True se quiser recomeçar do zero

anteriores = glob.glob(f"{RECORTES}/*.jpg")
if anteriores and not APAGAR_ANTERIORES:
    raise SystemExit(
        f"Já existem {len(anteriores)} recortes em {RECORTES}.\n"
        "Para recomeçar do zero, mude APAGAR_ANTERIORES para True nesta célula.\n"
        "Para aproveitar os que já estão lá, pule para a célula do CLIP.")
for velho in anteriores:
    os.remove(velho)

fontes = []
for padrao in ("images/train/*", "images/val/*"):
    fontes += glob.glob(os.path.join(DESTINO, padrao))
fontes = [f for f in fontes if f.lower().endswith((".jpg", ".jpeg", ".png"))]
print(f"{len(fontes)} imagens de origem")

MAX_RECORTES = 400
n = 0
for caminho in fontes:
    if n >= MAX_RECORTES:
        break
    img = cv2.imread(caminho)
    if img is None:
        continue
    r = meu_detector.predict(img, conf=.45, verbose=False)[0]
    for caixa in r.boxes.xyxy.cpu().numpy().astype(int):
        x1, y1, x2, y2 = caixa
        recorte = img[max(0, y1):y2, max(0, x1):x2]
        # recorte minúsculo não deixa ver a tampa: não serve para este problema
        if recorte.size == 0 or recorte.shape[0] < 96 or recorte.shape[1] < 32:
            continue
        cv2.imwrite(os.path.join(RECORTES, f"g{n:05d}.jpg"), recorte)
        n += 1
        if n >= MAX_RECORTES:
            break
print(f"{n} recortes em {RECORTES}")

In [ ]:
# ── CLIP dá o primeiro palpite em todos de uma vez ──
%pip install -q transformers
import torch, glob, os
from PIL import Image as PILImage
from transformers import CLIPProcessor, CLIPModel

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
clip = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(dispositivo)
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# As frases importam mais que o modelo. Descrever o SINAL VISUAL
# ("cápsula no gargalo", "gargalo vazio") funciona muito melhor do que
# repetir o rótulo ("lacrada", "aberta") — que não diz nada de visual.
FRASES = {
    "lacrada": [
        "a sealed wine bottle with foil capsule on the neck",
        "an unopened wine bottle with intact cork and seal",
    ],
    "aberta": [
        "an opened wine bottle with the cork removed, empty neck",
        "a wine bottle without capsule, open top",
    ],
}
rotulos = list(FRASES)
textos = [f for r in rotulos for f in FRASES[r]]
por_rotulo = [len(FRASES[r]) for r in rotulos]

arquivos = sorted(glob.glob(os.path.join(RECORTES, "*.jpg")))
print(f"classificando {len(arquivos)} recortes com CLIP…")

palpite, confianca = {}, {}
LOTE = 64
for i in range(0, len(arquivos), LOTE):
    lote = arquivos[i:i + LOTE]
    imgs = [PILImage.open(a).convert("RGB") for a in lote]
    entradas = proc(text=textos, images=imgs, return_tensors="pt",
                    padding=True).to(dispositivo)
    with torch.no_grad():
        probs = clip(**entradas).logits_per_image.softmax(dim=1).cpu()
    # junta as frases de cada rótulo (média), depois compara os dois lados
    corte, somas = 0, []
    for k in por_rotulo:
        somas.append(probs[:, corte:corte + k].mean(dim=1))
        corte += k
    somas = torch.stack(somas, dim=1)
    for arq, linha in zip(lote, somas):
        j = int(linha.argmax())
        palpite[arq] = rotulos[j]
        confianca[arq] = float(linha[j] / linha.sum())

from collections import Counter
print(Counter(palpite.values()))
print("pronto — agora é só revisar")

### A revisão

A grade abaixo mostra os recortes **ordenados do mais incerto para o mais
certo** — os erros do CLIP se concentram no começo, então revisar as primeiras
telas já resolve quase tudo.

Debaixo de cada imagem aparece o número e o palpite. **Anote os números que
estiverem errados** e jogue na célula seguinte.

In [ ]:
# ── grade de revisão ──
import matplotlib.pyplot as plt, cv2, math

POR_TELA = 24
TELA = 0        # ← 0, 1, 2… avance até se convencer

ordem = sorted(arquivos, key=lambda a: confianca[a])   # incerto primeiro
inicio = TELA * POR_TELA
pagina = ordem[inicio:inicio + POR_TELA]

if not pagina:
    print("acabou — não há mais recortes nesta tela")
else:
    colunas = 6
    linhas = math.ceil(len(pagina) / colunas)
    fig, axs = plt.subplots(linhas, colunas, figsize=(22, 4.2 * linhas))
    for k, (ax, arq) in enumerate(zip(axs.ravel(), pagina)):
        img = cv2.cvtColor(cv2.imread(arq), cv2.COLOR_BGR2RGB)
        ax.imshow(img); ax.axis("off")
        idx = inicio + k
        p, c = palpite[arq], confianca[arq]
        ax.set_title(f"[{idx}] {p} {c:.0%}", fontsize=17,
                     color=("#3fe0a8" if p == "lacrada" else "#ff5c5c"))
    for ax in axs.ravel()[len(pagina):]:
        ax.axis("off")
    fig.suptitle(f"tela {TELA} · revise e anote os ERRADOS", fontsize=28)
    plt.tight_layout(); plt.show()
    print(f"mostrando {inicio} a {inicio + len(pagina) - 1} de {len(ordem)}")

In [ ]:
# ── corrige os que o CLIP errou ──
# Cole os números que estavam errados. Ex.: ERRADOS = [0, 3, 7, 11]
ERRADOS = []

for idx in ERRADOS:
    arq = ordem[idx]
    palpite[arq] = "aberta" if palpite[arq] == "lacrada" else "lacrada"
    confianca[arq] = 1.0          # revisado por humano: confiança máxima

from collections import Counter
print(f"{len(ERRADOS)} corrigidos")
print(Counter(palpite.values()))

> Repita: mude `TELA` para 1, rode a grade, anote, corrija. Quando as
> classificações começarem a parecer todas certas, pode parar — dali para
> frente é o território de alta confiança do CLIP.

In [ ]:
# ── monta train/ e val/ automaticamente ──
import os, shutil, random
from collections import Counter

BASE = f"{DRIVE}/04-garrafas/treino"

# De novo: nada é apagado do seu Drive sem você mandar. Se já houver um
# dataset montado, o antigo é RENOMEADO com a data e hora — nunca perdido.
import time
ja_tem = any(os.path.isdir(f"{BASE}/{s}/{r}") and os.listdir(f"{BASE}/{s}/{r}")
             for s in ("train", "val") for r in rotulos
             if os.path.isdir(f"{BASE}/{s}/{r}"))
if ja_tem:
    guardado = f"{BASE}-anterior-{time.strftime('%Y%m%d-%H%M')}"
    shutil.move(BASE, guardado)
    print("dataset anterior guardado em:", guardado)

for split in ("train", "val"):
    for rot in rotulos:
        os.makedirs(f"{BASE}/{split}/{rot}", exist_ok=True)

porRotulo = {r: [] for r in rotulos}
for arq, rot in palpite.items():
    porRotulo[rot].append(arq)

resumo = Counter()
for rot, lista in porRotulo.items():
    random.shuffle(lista)
    corte = max(1, int(len(lista) * 0.2))          # 20% para validação
    for split, parte in (("val", lista[:corte]), ("train", lista[corte:])):
        for arq in parte:
            shutil.copy(arq, f"{BASE}/{split}/{rot}/{os.path.basename(arq)}")
            resumo[(split, rot)] += 1

print("dataset de classificação montado:\n")
for (split, rot), n in sorted(resumo.items()):
    print(f"  {split:<6} {rot:<9} {n:>4} imagens")

---
## Parte 5 · O especialista, treinado com o que a máquina rotulou

In [ ]:
from IPython.display import clear_output
import matplotlib.pyplot as plt

def treinar_mostrando(modelo, dados, epocas, imgsz=224, batch=32,
                      projeto="/content/runs", nome="ao_vivo", titulo="Aprendendo"):
    """Treina e redesenha a curva a cada epoca — o ponto alto da demo."""
    hist = {}                                  # epoca -> (perda, acuracia)

    def a_cada_epoca(trainer):
        m = getattr(trainer, "metrics", None) or {}
        acc = m.get("metrics/accuracy_top1")
        perda = float(trainer.loss.item()) if getattr(trainer, "loss", None) is not None else None
        hist[trainer.epoch + 1] = (perda, acc)   # dict: a epoca repetida sobrescreve

        eps = sorted(hist)
        perdas = [hist[e][0] for e in eps]
        accs = [(hist[e][1] or 0) * 100 for e in eps]

        clear_output(wait=True)
        fig, (a1, a2) = plt.subplots(1, 2, figsize=(20, 8))
        a1.plot(eps, perdas, lw=5, color="#ff5c5c", marker="o", ms=10)
        a1.set_title("ERRO — tem que descer"); a1.set_xlabel("época")
        a2.plot(eps, accs, lw=5, color="#3fe0a8", marker="o", ms=10)
        a2.set_ylim(0, 101)
        a2.set_title("ACERTO — tem que subir"); a2.set_xlabel("época"); a2.set_ylabel("%")
        if accs:
            a2.text(eps[-1], accs[-1], f"  {accs[-1]:.0f}%", fontsize=34,
                    color="#3fe0a8", va="center", fontweight="bold")
        fig.suptitle(f"{titulo} · época {max(eps)} de {epocas}", fontsize=34)
        plt.tight_layout(); plt.show()

    modelo.add_callback("on_fit_epoch_end", a_cada_epoca)
    r = modelo.train(data=dados, epochs=epocas, imgsz=imgsz, batch=batch,
                     project=projeto, name=nome, exist_ok=True, verbose=False, plots=True)
    print("pesos e graficos em:", r.save_dir)
    return r

In [ ]:
EPOCAS_CLS = 25

especialista = YOLO(f"{DRIVE}/00-pesos/yolo11n-cls.pt")
res_cls = treinar_mostrando(
    especialista, dados=BASE, epocas=EPOCAS_CLS,
    titulo="Aprendendo aberta × lacrada", nome="garrafas-cls",
)

import shutil
PESOS_CLS = f"{DRIVE}/04-garrafas/pesos/garrafas_best.pt"
shutil.copy(f"{res_cls.save_dir}/weights/best.pt", PESOS_CLS)
print("especialista salvo em", PESOS_CLS)

---
## Parte 6 · O pipeline completo

Detector treinado **+** especialista treinado, na mesma foto. É o inventário
que a palestra promete: quantas garrafas, e quantas de cada tipo.

In [ ]:
import cv2, glob, numpy as np, matplotlib.pyplot as plt
from collections import Counter

detector_final = YOLO(PESOS_DET)
classificador  = YOLO(PESOS_CLS)
CORES = {"lacrada": (168, 224, 63), "aberta": (92, 92, 255)}

fotos = [f for f in sorted(glob.glob(f"{DRIVE}/04-garrafas/inferencia/*"))
         if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp"))]
if not fotos:
    fotos = amostras[:1]          # cai para uma imagem do próprio dataset
FOTO = fotos[0]

img = cv2.imread(FOTO)
r = detector_final.predict(img, conf=.35, verbose=False)[0]
placar = Counter()
anotada = img.copy()

for caixa in r.boxes.xyxy.cpu().numpy().astype(int):
    x1, y1, x2, y2 = caixa
    recorte = img[max(0, y1):y2, max(0, x1):x2]
    if recorte.size == 0:
        continue
    p = classificador.predict(recorte, verbose=False)[0]
    rot = p.names[int(p.probs.top1)]
    conf = float(p.probs.top1conf)
    placar[rot] += 1
    cor = CORES.get(rot, (200, 200, 200))
    cv2.rectangle(anotada, (x1, y1), (x2, y2), cor, 4)
    cv2.putText(anotada, f"{rot} {conf:.0%}", (x1, max(26, y1 - 10)),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, cor, 3)

print("INVENTÁRIO")
for k, v in placar.most_common():
    print(f"  {v:>3}  {k}")

plt.figure(figsize=(18, 10))
plt.imshow(cv2.cvtColor(anotada, cv2.COLOR_BGR2RGB)); plt.axis("off")
plt.title(f"{sum(placar.values())} garrafas · " +
          " · ".join(f"{v} {k}" for k, v in placar.most_common()))
plt.tight_layout(); plt.show()
cv2.imwrite(f"{DRIVE}/04-garrafas/saida/inventario_completo.jpg", anotada)

### 🔴 O pipeline completo, ao vivo

O fecho: câmera aberta, cada garrafa detectada pelo **seu** detector,
classificada pelo **seu** especialista, e somada ao estoque.

In [ ]:
# ── motor de webcam ao vivo (leia o comentário: é o truque da demo) ──
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode, b64encode
import cv2, numpy as np, PIL.Image, io, time

def iniciar_webcam(largura=640, altura=480):
    # cria o video no navegador + a camada de overlay por cima dele
    display(Javascript('''
      var video, div = null, stream, imgElement, labelElement, captureCanvas;
      var pendingResolve = null, shutdown = false;
      var LARG = %d, ALT = %d;

      function removeDom() {
        if (stream) stream.getVideoTracks()[0].stop();
        if (video) video.remove();
        if (div) div.remove();
        video = null; div = null; stream = null;
        imgElement = null; captureCanvas = null; labelElement = null;
      }

      function onAnimationFrame() {
        if (!shutdown) window.requestAnimationFrame(onAnimationFrame);
        if (pendingResolve) {
          var result = "";
          if (!shutdown) {
            captureCanvas.getContext('2d').drawImage(video, 0, 0, LARG, ALT);
            result = captureCanvas.toDataURL('image/jpeg', 0.75);
          }
          var lp = pendingResolve;
          pendingResolve = null;
          lp(result);
        }
      }

      async function criarDom() {
        if (div !== null) return stream;

        div = document.createElement('div');
        div.style.border = '2px solid #3fe0a8';
        div.style.padding = '3px';
        div.style.width = '100%%';
        div.style.maxWidth = '900px';
        div.style.borderRadius = '10px';
        document.body.appendChild(div);

        var parar = document.createElement('div');
        parar.innerHTML = '&#9632; clique aqui para encerrar';
        parar.style.cssText = 'cursor:pointer;background:#3fe0a8;color:#06231a;' +
          'font-weight:700;padding:10px 16px;border-radius:8px;text-align:center;' +
          'font-family:system-ui,sans-serif;font-size:18px';
        div.appendChild(parar);
        parar.onclick = function() { shutdown = true; };

        video = document.createElement('video');
        video.style.display = 'block';
        video.style.width = '100%%';
        video.setAttribute('playsinline', '');
        video.onclick = function() { shutdown = true; };

        stream = await navigator.mediaDevices.getUserMedia(
          {video: {width: LARG, height: ALT}});
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();

        // a camada que recebe o resultado do modelo, por cima do vídeo
        imgElement = document.createElement('img');
        imgElement.style.position = 'absolute';
        imgElement.style.zIndex = 1;
        imgElement.style.pointerEvents = 'none';
        imgElement.onclick = function() { shutdown = true; };
        div.appendChild(imgElement);

        labelElement = document.createElement('div');
        labelElement.style.cssText = 'font-family:system-ui,sans-serif;' +
          'font-size:20px;color:#e6edf3;padding:8px 4px';
        div.appendChild(labelElement);

        captureCanvas = document.createElement('canvas');
        captureCanvas.width = LARG;
        captureCanvas.height = ALT;
        window.requestAnimationFrame(onAnimationFrame);

        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        return stream;
      }

      async function quadro(rotulo, overlay) {
        if (shutdown) { removeDom(); shutdown = false; return ''; }
        stream = await criarDom();
        if (rotulo != "") labelElement.innerHTML = rotulo;
        if (overlay != "") {
          var r = video.getClientRects()[0];
          imgElement.style.top = r.top + "px";
          imgElement.style.left = r.left + "px";
          imgElement.style.width = r.width + "px";
          imgElement.style.height = r.height + "px";
          imgElement.src = overlay;
        }
        var result = await new Promise(function(resolve) { pendingResolve = resolve; });
        shutdown = false;
        return {'img': result};
      }
    ''' % (largura, altura)))


def _para_imagem(resposta):
    # base64 do navegador -> imagem BGR do OpenCV
    if not resposta:
        return None
    dados = b64decode(resposta.split(',')[1])
    arr = np.frombuffer(dados, dtype=np.uint8)
    return cv2.imdecode(arr, flags=1)


def _para_overlay(rgba):
    # array RGBA -> data URI PNG, para o navegador sobrepor ao video
    img = PIL.Image.fromarray(rgba, 'RGBA')
    buf = io.BytesIO()
    img.save(buf, format='png')
    return 'data:image/png;base64,' + b64encode(buf.getvalue()).decode('utf-8')


def rodar_ao_vivo(processa, largura=640, altura=480, rotulo_inicial='iniciando…'):
    # Laco principal.
    #
    # `processa(frame_bgr, overlay_rgba)` recebe o quadro e uma tela RGBA
    # transparente do mesmo tamanho, desenha nela, e devolve o texto do
    # painel. Encerre clicando no botão verde (ou no próprio vídeo).
    iniciar_webcam(largura, altura)
    overlay = np.zeros([altura, largura, 4], dtype=np.uint8)
    envio = ''
    rotulo = rotulo_inicial
    n = 0
    t0 = time.time()
    try:
        while True:
            resposta = eval_js('quadro("{}", "{}")'.format(rotulo, envio))
            if not resposta:
                break
            frame = _para_imagem(resposta['img'])
            if frame is None:
                break

            overlay[:] = 0
            texto = processa(frame, overlay)

            n += 1
            fps = n / max(1e-6, time.time() - t0)
            rotulo = '{} &nbsp;·&nbsp; {:.1f} quadros/s'.format(texto, fps)
            envio = _para_overlay(overlay)
    except Exception as e:
        print('encerrado:', type(e).__name__, e)
    print('fim · {} quadros processados'.format(n))

In [ ]:
import cv2, numpy as np

estoque_tipo = {}      # id -> "lacrada" | "aberta"

def processa_pipeline(frame, overlay):
    h, w = frame.shape[:2]
    r = detector_final.track(frame, persist=True, conf=.35, verbose=False, imgsz=480)[0]

    na_cena = 0
    if r.boxes is not None and len(r.boxes):
        na_cena = len(r.boxes)
        ids = (r.boxes.id.cpu().numpy().astype(int) if r.boxes.id is not None
               else np.arange(na_cena))
        for gid, caixa in zip(ids, r.boxes.xyxy.cpu().numpy().astype(int)):
            gid = int(gid)
            x1, y1, x2, y2 = caixa
            recorte = frame[max(0, y1):y2, max(0, x1):x2]
            if recorte.size == 0:
                continue
            # classifica uma vez por garrafa: reclassificar todo quadro
            # faz o rótulo piscar entre os dois e parece defeito
            if gid not in estoque_tipo:
                p = classificador.predict(recorte, verbose=False)[0]
                estoque_tipo[gid] = p.names[int(p.probs.top1)]
            rot = estoque_tipo[gid]
            cor = (168, 224, 63, 255) if rot == "lacrada" else (92, 92, 255, 255)
            cv2.rectangle(overlay, (x1, y1), (x2, y2), cor, 3)
            cv2.rectangle(overlay, (x1, max(0, y1 - 32)), (x1 + 190, y1), cor, -1)
            cv2.putText(overlay, "#{} {}".format(gid, rot), (x1 + 6, max(22, y1 - 9)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (10, 15, 20, 255), 2)

    lacradas = sum(1 for v in estoque_tipo.values() if v == "lacrada")
    abertas  = sum(1 for v in estoque_tipo.values() if v == "aberta")

    cv2.rectangle(overlay, (0, 0), (w, 96), (13, 17, 23, 215), -1)
    cv2.putText(overlay, "ESTOQUE", (16, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (125, 133, 144, 255), 2)
    cv2.putText(overlay, str(len(estoque_tipo)), (16, 84), cv2.FONT_HERSHEY_SIMPLEX, 1.9, (230, 237, 243, 255), 4)
    cv2.putText(overlay, "LACRADAS", (int(w * .34), 30), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (125, 133, 144, 255), 2)
    cv2.putText(overlay, str(lacradas), (int(w * .34), 84), cv2.FONT_HERSHEY_SIMPLEX, 1.9, (168, 224, 63, 255), 4)
    cv2.putText(overlay, "ABERTAS", (int(w * .66), 30), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (125, 133, 144, 255), 2)
    cv2.putText(overlay, str(abertas), (int(w * .66), 84), cv2.FONT_HERSHEY_SIMPLEX, 1.9, (92, 92, 255, 255), 4)

    return "estoque <b>{}</b> &nbsp;·&nbsp; {} lacradas, {} abertas".format(
        len(estoque_tipo), lacradas, abertas)

estoque_tipo.clear()
rodar_ao_vivo(processa_pipeline, largura=640, altura=480,
              rotulo_inicial='mostre garrafas para a câmera…')

In [ ]:
estoque_tipo.clear()
print("estoque zerado")

---

## O que dizer no palco

> *"Esse sistema não existia hoje de manhã. As imagens vieram de um banco
> público. Quem separou aberta de lacrada foi outro modelo de IA — eu só
> corrigi os erros dele. E o resultado é um contador de estoque que roda na
> câmera do meu notebook."*

E o encerramento honesto, que vale mais que a demo:

> *"Isso aqui não é o estado da arte. É o que dá para montar numa tarde,
> de graça, com ferramenta aberta. O estado da arte está muito além — e é
> por isso que a pergunta não é se cabe no seu negócio. É quando."*

## Mais imagens, se quiser

- **Open Images V7** — <https://storage.googleapis.com/openimages/web/index.html>
  (classes `Bottle`, `Wine`, `Beer`, `Drink`)
- **Roboflow Universe** — <https://universe.roboflow.com>
  (busque *wine bottle*, *retail shelf*, *supermarket*)

> **Licença:** Open Images é CC-BY. Datasets do Roboflow variam — a licença
> aparece na página de cada um. Para palestra não muda nada; para produto de
> cliente, confira.